# 증강 유/무 비교 실험

동일한 모델(EfficientNet-B0)과 하이퍼파라미터로  
**증강 없음** vs **증강 있음** 두 버전을 순서대로 학습하고 결과를 비교한다.

## 0. 공통 설정

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tqdm import tqdm

# ── 재현성 고정 ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {DEVICE}')

# ── 하이퍼파라미터 ────────────────────────────────────────
DATA_DIR    = './raw2'
SAVE_DIR    = './checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 30
LR          = 1e-4
VAL_RATIO   = 0.2
NUM_WORKERS = 0

CLASSES     = ['mug', 'straight', 'taper_smooth', 'taper_step']
NUM_CLASSES = len(CLASSES)

## 1. Transform 정의

세 가지 transform을 준비한다.

| 이름 | 용도 |
|------|------|
| `plain_transform` | 증강 없음 버전의 train용 (리사이즈 + 정규화만) |
| `aug_transform` | 증강 있음 버전의 train용 (다양한 증강 적용) |
| `val_transform` | 두 버전 공통 val용 (리사이즈 + 정규화만) |

In [ ]:
NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std =[0.229, 0.224, 0.225])

# 증강 없음 (train)
plain_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    NORMALIZE,
])

# 증강 있음 (train)
aug_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    NORMALIZE,
])

# 공통 val
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    NORMALIZE,
])

print('Transform 정의 완료')

## 2. 공통 함수 정의

In [ ]:
def get_loaders(train_transform, val_transform):
    """주어진 transform으로 train/val DataLoader 반환"""
    # 분할 인덱스는 두 버전이 동일해야 공정한 비교 가능
    base = datasets.ImageFolder(root=DATA_DIR, transform=val_transform)
    val_size   = int(len(base) * VAL_RATIO)
    train_size = len(base) - val_size
    train_ds, val_ds = random_split(
        base, [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )
    # train에만 train_transform 적용
    train_ds.dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, val_ds


def build_model():
    """매번 새로운 pretrained 모델 반환 (두 버전이 동일한 초기 가중치에서 시작)"""
    torch.manual_seed(SEED)
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


def run_experiment(label, train_transform, val_transform):
    """한 버전 전체 학습 실행 → history 반환"""
    print(f'\n{'='*60}')
    print(f'  실험: {label}')
    print(f'{'='*60}')

    train_loader, val_loader, val_ds = get_loaders(train_transform, val_transform)
    model     = build_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    best_acc, best_state = 0.0, None

    print(f'  {'Epoch':>5}  {'Tr Loss':>8}  {'Tr Acc':>7}  {'Va Loss':>8}  {'Va Acc':>7}')
    print(f'  {'-'*45}')

    for epoch in tqdm(range(1, NUM_EPOCHS + 1), desc=f'  {label}'):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)

        if va_acc > best_acc:
            best_acc   = va_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    save_path = os.path.join(SAVE_DIR, f'best_{label.replace(" ", "_")}.pth')
    torch.save(best_state, save_path)
    print(f'\n  ✅ Best Val Acc: {best_acc*100:.2f}%  →  {save_path}')

    # val 예측 (confusion matrix용)
    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_labels = [], []
    val_loader2, _ = DataLoader(val_ds, batch_size=BATCH_SIZE), None
    with torch.no_grad():
        for imgs, labels in DataLoader(val_ds, batch_size=BATCH_SIZE):
            preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    return history, best_acc, all_preds, all_labels


print('공통 함수 정의 완료')

## 3. 실험 실행

두 버전을 순서대로 학습한다. GPU 환경에서 각 약 3~5분 소요.

In [ ]:
# ── 버전 1: 증강 없음 ────────────────────────────────────
hist_plain, best_plain, preds_plain, labels_plain = run_experiment(
    label           = '증강 없음',
    train_transform = plain_transform,
    val_transform   = val_transform
)

# ── 버전 2: 증강 있음 ────────────────────────────────────
hist_aug, best_aug, preds_aug, labels_aug = run_experiment(
    label           = '증강 있음',
    train_transform = aug_transform,
    val_transform   = val_transform
)

print('\n두 실험 완료!')

## 4. 학습 곡선 비교

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('증강 유/무 학습 결과 비교', fontsize=15, fontweight='bold')

configs = [
    (axes[0][0], 'train_loss', 'Train Loss'),
    (axes[0][1], 'val_loss',   'Val Loss'),
    (axes[1][0], 'train_acc',  'Train Accuracy (%)'),
    (axes[1][1], 'val_acc',    'Val Accuracy (%)'),
]

for ax, key, title in configs:
    is_acc = 'acc' in key
    p_vals = [v*100 if is_acc else v for v in hist_plain[key]]
    a_vals = [v*100 if is_acc else v for v in hist_aug[key]]
    ax.plot(epochs, p_vals, label='증강 없음', color='#E74C3C', linestyle='--')
    ax.plot(epochs, a_vals, label='증강 있음', color='#2ECC71')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'compare_curves.png'), dpi=150)
plt.show()

## 5. 최종 성능 요약

In [ ]:
print('┌─────────────────────────────────────────┐')
print('│          최종 Val Accuracy 비교          │')
print('├─────────────────────────────────────────┤')
print(f'│  증강 없음  :  {best_plain*100:6.2f}%                   │')
print(f'│  증강 있음  :  {best_aug*100:6.2f}%                   │')
diff = (best_aug - best_plain) * 100
sign = '+' if diff >= 0 else ''
print(f'│  차이       :  {sign}{diff:.2f}%p                  │')
print('└─────────────────────────────────────────┘')

winner = '증강 있음' if best_aug >= best_plain else '증강 없음'
print(f'\n→ 더 좋은 버전: [{winner}]')

## 6. Confusion Matrix 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, preds, labels_gt, title in zip(
    axes,
    [preds_plain, preds_aug],
    [labels_plain, labels_aug],
    ['증강 없음', '증강 있음']
):
    cm = confusion_matrix(labels_gt, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(f'Confusion Matrix — {title}')
    ax.set_ylabel('실제')
    ax.set_xlabel('예측')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'compare_cm.png'), dpi=150)
plt.show()

## 7. Classification Report 비교

In [ ]:
print('=== 증강 없음 ===')
print(classification_report(labels_plain, preds_plain, target_names=CLASSES))

print('=== 증강 있음 ===')
print(classification_report(labels_aug, preds_aug, target_names=CLASSES))